# Mangrove attribution vs protected status

This notebook loads mangrove-level attributed avoided EAD results and overlays them with protected areas to estimate:
- the proportion of benefited mangroves under protected status vs not protected
- the protected vs unprotected share by **count**, **area**, and **attributed benefit (USD)**


In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# Paths and settings
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
jamaica_metric_grid_crs = 'EPSG:3448'

attribution_gpkg_path = base_path / 'dphil_paper_3/results/damage_estimates/mangrove_attribution/mangrove_attribution_total_1000m.gpkg'
forest_reserves_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/protected_landcover/forest_reserves.shp'
protected_areas_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/protected_landcover/protected_areas.shp'

benefit_column = 'Total_Avoided_EAD_USD_attributed'
benefit_threshold_usd = 0.0  # benefited mangroves are > this value

for p in [attribution_gpkg_path, forest_reserves_path, protected_areas_path]:
    if not p.exists():
        raise FileNotFoundError(f'Missing input file: {p}')

print('Attribution file:', attribution_gpkg_path)
print('Forest reserves file:', forest_reserves_path)
print('Protected areas file:', protected_areas_path)


In [ ]:
# Load data
mangrove_attr = gpd.read_file(attribution_gpkg_path).to_crs(jamaica_metric_grid_crs)
forest_reserves = gpd.read_file(forest_reserves_path).to_crs(jamaica_metric_grid_crs)
protected_areas = gpd.read_file(protected_areas_path).to_crs(jamaica_metric_grid_crs)

if benefit_column not in mangrove_attr.columns:
    raise KeyError(f"Column '{benefit_column}' not found in attribution layer. Available: {list(mangrove_attr.columns)}")

# Combine protected layers (geometry only), clean, dissolve to one geometry
protected_all = gpd.GeoDataFrame(
    pd.concat([forest_reserves[['geometry']], protected_areas[['geometry']]], ignore_index=True),
    crs=jamaica_metric_grid_crs,
)
protected_all = protected_all[protected_all.geometry.notna()].copy()
if hasattr(protected_all.geometry, 'make_valid'):
    protected_all['geometry'] = protected_all.geometry.make_valid()
else:
    protected_all['geometry'] = protected_all.buffer(0)

try:
    protected_union = protected_all.geometry.union_all()
except Exception:
    protected_union = protected_all.unary_union

combined_protected_layers = gpd.GeoDataFrame(geometry=[protected_union], crs=jamaica_metric_grid_crs)

print(f'Mangrove attribution rows: {len(mangrove_attr):,}')
print(f'Forest reserve polygons: {len(forest_reserves):,}')
print(f'Protected area polygons: {len(protected_areas):,}')


In [ ]:
# Select mangroves with attributed benefits and compute overlap metrics
benefited = mangrove_attr[mangrove_attr[benefit_column].fillna(0.0) > benefit_threshold_usd].copy()

if len(benefited) == 0:
    raise ValueError('No mangroves found above benefit_threshold_usd. Lower the threshold or check attribution file.')

if hasattr(benefited.geometry, 'make_valid'):
    benefited['geometry'] = benefited.geometry.make_valid()
else:
    benefited['geometry'] = benefited.buffer(0)

benefited['benefited_area_m2'] = benefited.geometry.area
benefited['protected_area_m2'] = benefited.geometry.intersection(protected_union).area
benefited['protected_area_m2'] = benefited['protected_area_m2'].fillna(0.0).clip(lower=0.0)
benefited['unprotected_area_m2'] = (benefited['benefited_area_m2'] - benefited['protected_area_m2']).clip(lower=0.0)
benefited['is_protected_polygon'] = benefited['protected_area_m2'] > 0

# Allocate attributed benefit by area share (partial overlaps handled)
area_share = np.where(benefited['benefited_area_m2'] > 0, benefited['protected_area_m2'] / benefited['benefited_area_m2'], 0.0)
benefited['benefit_protected_usd'] = benefited[benefit_column] * area_share
benefited['benefit_unprotected_usd'] = benefited[benefit_column] - benefited['benefit_protected_usd']

# Summary metrics
count_total = int(len(benefited))
count_protected = int(benefited['is_protected_polygon'].sum())
count_unprotected = count_total - count_protected

area_total = float(benefited['benefited_area_m2'].sum())
area_protected = float(benefited['protected_area_m2'].sum())
area_unprotected = float(benefited['unprotected_area_m2'].sum())

benefit_total = float(benefited[benefit_column].sum())
benefit_protected = float(benefited['benefit_protected_usd'].sum())
benefit_unprotected = float(benefited['benefit_unprotected_usd'].sum())

summary = pd.DataFrame([
    {
        'Metric': 'Mangrove polygons (count)',
        'Protected': count_protected,
        'Not_Protected': count_unprotected,
        'Total': count_total,
        'Protected_%': 100.0 * count_protected / count_total if count_total else np.nan,
        'Not_Protected_%': 100.0 * count_unprotected / count_total if count_total else np.nan,
    },
    {
        'Metric': 'Benefited mangrove area (ha)',
        'Protected': area_protected / 10000.0,
        'Not_Protected': area_unprotected / 10000.0,
        'Total': area_total / 10000.0,
        'Protected_%': 100.0 * area_protected / area_total if area_total else np.nan,
        'Not_Protected_%': 100.0 * area_unprotected / area_total if area_total else np.nan,
    },
    {
        'Metric': 'Attributed avoided EAD (USD)',
        'Protected': benefit_protected,
        'Not_Protected': benefit_unprotected,
        'Total': benefit_total,
        'Protected_%': 100.0 * benefit_protected / benefit_total if benefit_total else np.nan,
        'Not_Protected_%': 100.0 * benefit_unprotected / benefit_total if benefit_total else np.nan,
    },
])

summary


In [ ]:
# Sense check: total mangroves (all polygons) protected vs not protected
all_mangroves = mangrove_attr.copy()
if hasattr(all_mangroves.geometry, 'make_valid'):
    all_mangroves['geometry'] = all_mangroves.geometry.make_valid()
else:
    all_mangroves['geometry'] = all_mangroves.buffer(0)

all_mangroves['total_area_m2'] = all_mangroves.geometry.area
all_mangroves['protected_area_m2'] = all_mangroves.geometry.intersection(protected_union).area
all_mangroves['protected_area_m2'] = all_mangroves['protected_area_m2'].fillna(0.0).clip(lower=0.0)
all_mangroves['unprotected_area_m2'] = (all_mangroves['total_area_m2'] - all_mangroves['protected_area_m2']).clip(lower=0.0)
all_mangroves['is_protected_polygon'] = all_mangroves['protected_area_m2'] > 0

all_count_total = int(len(all_mangroves))
all_count_protected = int(all_mangroves['is_protected_polygon'].sum())
all_count_unprotected = all_count_total - all_count_protected

all_area_total = float(all_mangroves['total_area_m2'].sum())
all_area_protected = float(all_mangroves['protected_area_m2'].sum())
all_area_unprotected = float(all_mangroves['unprotected_area_m2'].sum())

total_vs_benefited = pd.DataFrame([
    {
        'Group': 'All mangrove polygons',
        'Total_count': all_count_total,
        'Protected_count': all_count_protected,
        'Not_protected_count': all_count_unprotected,
        'Protected_count_%': 100.0 * all_count_protected / all_count_total if all_count_total else np.nan,
        'Total_area_ha': all_area_total / 10000.0,
        'Protected_area_ha': all_area_protected / 10000.0,
        'Not_protected_area_ha': all_area_unprotected / 10000.0,
        'Protected_area_%': 100.0 * all_area_protected / all_area_total if all_area_total else np.nan,
    },
    {
        'Group': f'Benefited polygons ({benefit_column} > {benefit_threshold_usd})',
        'Total_count': count_total,
        'Protected_count': count_protected,
        'Not_protected_count': count_unprotected,
        'Protected_count_%': 100.0 * count_protected / count_total if count_total else np.nan,
        'Total_area_ha': area_total / 10000.0,
        'Protected_area_ha': area_protected / 10000.0,
        'Not_protected_area_ha': area_unprotected / 10000.0,
        'Protected_area_%': 100.0 * area_protected / area_total if area_total else np.nan,
    },
])

total_vs_benefited


In [ ]:
# Quick lookup: top benefited mangroves and their protected status
top_cols = [c for c in ['Mangrove_ID', 'Parish', 'HECTARES', benefit_column, 'is_protected_polygon', 'protected_area_m2', 'benefited_area_m2'] if c in benefited.columns]
benefited[top_cols].sort_values(by=benefit_column, ascending=False).head(20)


In [ ]:
# Map: benefited mangroves by protected status, with protected area outlines
fig, ax = plt.subplots(figsize=(12, 10))

# optional backdrop: all attributed mangroves
mangrove_attr.plot(ax=ax, color='lightgrey', edgecolor='none', alpha=0.35, label='All attribution mangroves')

protected_outline = combined_protected_layers.boundary
protected_outline.plot(ax=ax, color='navy', linewidth=0.8, alpha=0.9, label='Protected boundaries')

benefited[benefited['is_protected_polygon']].plot(
    ax=ax,
    color='#1a9850',
    edgecolor='black',
    linewidth=0.2,
    alpha=0.9,
    label='Benefited mangroves (protected)',
)
benefited[~benefited['is_protected_polygon']].plot(
    ax=ax,
    color='#d73027',
    edgecolor='black',
    linewidth=0.2,
    alpha=0.9,
    label='Benefited mangroves (not protected)',
)

ax.set_title('Mangroves with attributed avoided EAD: protected vs not protected', fontsize=13)
ax.set_axis_off()
ax.legend(loc='lower left', frameon=True)
plt.tight_layout()
plt.show()


In [ ]:
# Optional: save outputs next to attribution results
out_dir = attribution_gpkg_path.parent / 'protected_status_overlay'
out_dir.mkdir(parents=True, exist_ok=True)

summary.to_csv(out_dir / 'mangrove_protected_status_summary.csv', index=False)
benefited.to_file(out_dir / 'benefited_mangroves_with_protected_status.gpkg', driver='GPKG')

print('Saved summary:', out_dir / 'mangrove_protected_status_summary.csv')
print('Saved layer:', out_dir / 'benefited_mangroves_with_protected_status.gpkg')
